# AI Workforce Investment Planner

**Project:** AI Workforce Investment Planner  
**Student:** Hehua (Leo) Liu - confirmation pending  
**Branch:** `Leo`  
**Branch URL:** [https://github.com/Hardcore-LEO/dso576-project/tree/Leo](https://github.com/Hardcore-LEO/dso576-project/tree/Leo)  
**Submitted commit:** `4993964fc785f73b9c518b49223af6b6d6192f68`

**Business question:** The Executive Director of the California Workforce Development Board must decide which California occupation × region combinations should be prioritized for AI upskilling and workforce-transition support.

The output is a transparent screening shortlist for workforce-planning review. It is not an automatic funding decision, and higher AI exposure does not automatically mean that a job should be automated.

## Workflow

1. Inspect each original source.
2. Apply documented, source-specific cleaning rules.
3. Show cleaned records and reconcile row counts.
4. Integrate California occupation × region records.
5. Validate the final analysis and five predeclared records.

# Inspect and clean the data: 14-item checklist

## 1 State the project vision and question

**Status: Done**  
**Evidence or explanation:** The project produces a transparent California workforce-planning shortlist. The main comparison is the within-region screening score built from observed AI exposure, employment size, annual wage, and national employment outlook. See **Business question** and **Screening score**.

## 2 Define one row

**Status: Done**  
**Evidence or explanation:** In the final analysis, one row represents one detailed occupation in one California metropolitan or nonmetropolitan region. `AREA` and `OCC_CODE`, together with the retained OEWS grain fields, identify the record. See **Data integration and merge verification** and **Duplicate checks**.

## 3 Preserve the source

**Status: Done**  
**Evidence or explanation:** The five original files remain unchanged. File names, sizes, and SHA-256 hashes are shown in **Source inventory**, and the pipeline confirms that hashes are unchanged after cleaning.

## 4 Inspect columns and types

**Status: Done**  
**Evidence or explanation:** Dates and numeric fields are converted with explicit rules. Failed conversion counts are reported after each source and are zero for the declared conversions. See each source's **Original data checks**, **Cleaning rules**, and **Row-count reconciliation**.

## 5 Standardize text

**Status: Done**  
**Evidence or explanation:** Only outside whitespace is removed from display text. Meaningful spelling, punctuation, capitalization, and leading zeros in identifiers are preserved. Raw occupation titles are retained in projection output columns ending in `__raw`.

## 6 Investigate duplicates

**Status: Done**  
**Evidence or explanation:** Exact duplicate rows and repeated keys are checked separately. The results are displayed in **Final validation - Duplicate checks**. Valid exposure SOC codes are unique.

## 7 Handle missing data

**Status: Done**  
**Evidence or explanation:** Missing counts before and after cleaning are displayed in **Missing values before and after cleaning**. OEWS markers such as `**`, `*`, and `#` become missing numeric values with separate status columns; they are never changed to zero.

## 8 Check suspicious values

**Status: Done**  
**Evidence or explanation:** Suspicious values are flagged before treatment. Thirty-two malformed SOC strings are preserved and made join-ineligible. The Claude value 199.11 is retained because it belongs to a time metric, and negative projection growth remains negative.

## 9 Verify merges

**Status: Done**  
**Evidence or explanation:** Both joins use exact SOC codes and are validated as many-to-one. **Join and checksum verification** reports 10,145 exposure matches, 1,702 exposure nonmatches, 11,847 projection matches, and zero projection nonmatches.

## 10 Reconcile changes

**Status: Done**  
**Evidence or explanation:** **Changes after each major transformation** reports starting rows, final rows, exclusions, changed cells, flags, failed conversions, and explanations. The employment checksum remains 17,277,510 before and after joins.

## 11 Check calculations

**Status: Done**  
**Evidence or explanation:** Scores are calculated only when all four inputs are present; missing components are not treated as zero or reweighted. There are 9,829 eligible rows and 2,018 retained incomplete rows. See **Screening score**.

## 12 Verify individual records

**Status: Done**  
**Evidence or explanation:** **Five predeclared record checks** compares original, expected, and actual values for five difficult records. All five checks pass.

## 13 Test reproducibility

**Status: Done**  
**Evidence or explanation:** `python execute_notebook.py` rebuilds and runs the notebook from the original data. `python -m unittest -v test_clean_data.py` runs six automated tests. The executed notebook confirms that every pipeline assertion passed.

## 14 Document remaining limitations

**Status: Done**  
**Evidence or explanation:** The **Remaining limitations** section explains unresolved malformed codes, geographic differences between sources, missing estimates, and the judgment involved in screening weights.

## Setup and reproducible pipeline

The notebook uses basic pandas commands for inspection. The longer cleaning functions are stored in `clean_data.py` so that the notebook stays readable and the same rules can be rerun.

In [1]:
from pathlib import Path
import json
import pandas as pd
import clean_data

ROOT = Path.cwd()
SOURCE = ROOT.parent
OUTPUT = ROOT / "outputs"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

validation = clean_data.main(show_summary=False)
stage_changes = pd.read_csv(OUTPUT / "stage_changes.csv")
missing_counts = pd.read_csv(OUTPUT / "missing_counts.csv")
duplicate_checks = pd.read_csv(OUTPUT / "duplicates.csv")

print("Pipeline completed successfully.")
print("All validation assertions passed:", validation["all_assertions_passed"])
print("Original source hashes unchanged:", validation["source_hashes_unchanged"])

Pipeline completed successfully.
All validation assertions passed: True
Original source hashes unchanged: True


## Source inventory

These are the five input files used by the cleaning pipeline. No file outside the supplied `clean_data` folder is used as a data source.

- Anthropic Economic Index snapshot: `aei_claude_ai_2026-06-26.csv` (2026-06-26 snapshot).
- Course-provided occupation exposure file: `job_exposure.csv` (no separate version date stated in the supplied file).
- BLS OEWS metropolitan and nonmetropolitan estimates: May 2025 releases.
- BLS Employment Projections workbook: 2025-2035 projection period.

In [2]:
source_inventory = pd.read_json(OUTPUT / "source_manifest.json")
source_inventory[["source_file", "bytes", "sha256"]]

,source_file,bytes,sha256
0,aei_claude_ai_2026-06-26.csv,219174671,f974b358bce0e5a8417510c61da4342234cd0de9d9d0b62acf4c6dbcf8ec7b68
1,job_exposure.csv,37079,94c8998796891efc0ebe378252e3eba4db330f054ee19d337fdfce992a5f727e
2,BOS_M2025_dl.xlsx,9891462,2a06fc23e9c3d487d767b76f0efa749252f4ad55f745640064845aa2c4b1aa1b
3,MSA_M2025_dl.xlsx,30628700,faf00eee127763c8c082d8fb862717ef2ad3a75f16f8975dc1e70935f27bcbb0
4,occupation.XLSX,413398,a7d060f0576f7d4f5b829b9a0bb4a6c11a12766d159ebb0e07aab5bd2e7583c5


# 1. Anthropic Economic Index

This source contains Claude usage metrics by date, geography, category, hierarchy level, and task or occupation node. Metric meaning depends on `metric_id`, so the values cannot all be treated as percentages.

## Original data checks

The first five source rows are displayed before cleaning. The audit checks row count, duplicates, date conversion, numeric conversion, and metric units.

In [3]:
claude_raw = pd.read_csv(SOURCE / "aei_claude_ai_2026-06-26.csv", nrows=5)
print("Full source rows:", f'{validation["claude"]["rows"]:,}')
claude_raw

Full source rows: 1,636,573


,date_start,date_end,geo_id,geo_level,category_name,hierarchy_level,metric_id,value,node_name,node_external_id
0,2026-05-01,2026-06-01,ZA-WC,subregion,onet,2,pct,0.25,"Prepare financial documents, reports, or budgets.",4.A.3.b.6.I01
1,2026-05-01,2026-06-01,ZA-WC,subregion,onet,0,pct,0.40,Locate unusual or unique information in response to specific requests.,22420
2,2026-05-01,2026-06-01,ZA-WC,subregion,onet,0,pct,0.68,"Answer customers' questions, and provide information on procedures or policies.",2408
3,2026-05-01,2026-06-01,ZA-KZN,subregion,onet,1,pct,0.94,Advise others on business or operational matters.,4.A.4.b.6.I05.D10
4,2026-05-01,2026-06-01,ZA-KZN,subregion,onet,1,pct,2.18,Tutor students who need extra assistance.,4.A.4.b.3.I02.D10


In [4]:
pd.DataFrame({
    "check": [
        "Exact duplicate excess",
        "Declared-key duplicate excess",
        "Failed date conversions",
        "Failed numeric conversions",
        "Suspicious values after metric-aware checks",
    ],
    "result": [
        validation["claude"]["exact_duplicate_excess"],
        validation["claude"]["duplicate_key_excess"],
        validation["claude"]["failed_dates"],
        validation["claude"]["failed_numeric"],
        validation["claude"]["suspicious_values"],
    ],
})

,check,result
0,Exact duplicate excess,0
1,Declared-key duplicate excess,0
2,Failed date conversions,0
3,Failed numeric conversions,0
4,Suspicious values after metric-aware checks,0


## Cleaning rules

- Preserve every source row and add `_source_file` and `_source_row`.
- Parse the two date columns as dates.
- Convert hierarchy level and value with `pd.to_numeric()`.
- Assign units from `metric_id` instead of applying one universal range.
- Flag a value only when it violates the rule for its own metric.

## Cleaned data preview

In [5]:
claude_clean = pd.read_csv(OUTPUT / "cleaned_claude_metrics.csv.gz", nrows=8)
claude_clean[[
    "_source_row", "date_start", "geo_level", "category_name",
    "metric_id", "value", "metric_unit", "suspicious_value"
]]

,_source_row,date_start,geo_level,category_name,metric_id,value,metric_unit,suspicious_value
0,2,2026-05-01,subregion,onet,pct,0.25,percentage points,False
1,3,2026-05-01,subregion,onet,pct,0.40,percentage points,False
2,4,2026-05-01,subregion,onet,pct,0.68,percentage points,False
3,5,2026-05-01,subregion,onet,pct,0.94,percentage points,False
4,6,2026-05-01,subregion,onet,pct,2.18,percentage points,False
5,7,2026-05-01,subregion,onet,pct,3.24,percentage points,False
6,8,2026-05-01,subregion,onet,pct,1.64,percentage points,False
7,9,2026-05-01,subregion,onet,pct,2.12,percentage points,False


## Row-count reconciliation

In [6]:
stage_changes.loc[stage_changes["stage"] == "claude", [
    "stage", "input_rows", "output_rows", "rows_excluded",
    "changed_cells", "new_flags", "summary"
]]

,stage,input_rows,output_rows,rows_excluded,changed_cells,new_flags,summary
0,claude,1636573,1636573,0,6546292,0,"Parsed dates, hierarchy level, and value; added metric units and metric-aware flags. No rows removed."


**Summary:** All Claude rows are retained. Dates and numeric values convert successfully, the declared key is unique, and metric-specific units prevent a valid time value such as 199.11 from being incorrectly flagged as a percentage.

# 2. Occupation AI-exposure data

This source supplies observed AI exposure by occupation. A valid join key must follow the six-digit SOC pattern `NN-NNNN`.

## Original data checks

In [7]:
exposure_raw = pd.read_csv(SOURCE / "job_exposure.csv")
valid_soc = exposure_raw["occ_code"].astype(str).str.fullmatch(r"\d{2}-\d{4}")
print("Shape:", exposure_raw.shape)
print("Valid SOC codes:", int(valid_soc.sum()))
print("Malformed codes:", int((~valid_soc).sum()))
print("Exact duplicate rows:", int(exposure_raw.duplicated().sum()))
exposure_raw.head()

Shape: (756, 3)
Valid SOC codes: 724
Malformed codes: 32
Exact duplicate rows: 0


,occ_code,title,observed_exposure
0,11-1011,Chief Executives,0.0333
1,11-1021,General and Operations Managers,0.1378
2,11-1031,Legislators,0.0000
3,Nov-11,Advertising and Promotions Managers,0.1731
4,Nov-21,Marketing Managers,0.3195


## Cleaning rules

- Preserve every row and the original occupation-code text.
- Trim outside spaces from text fields.
- Convert exposure to numeric without changing its unit.
- Mark malformed SOC codes as not eligible for exact-code joins.
- Do not guess repairs for date-like values such as `Nov-11`.

## Cleaned data preview

In [8]:
exposure_clean = pd.read_csv(OUTPUT / "cleaned_job_exposure.csv")
exposure_clean.head(8)

,_source_file,_source_row,occ_code,title,observed_exposure,occ_code_raw,code_format_valid,code_join_eligible,suspicious_value
0,job_exposure.csv,2,11-1011,Chief Executives,0.0333,11-1011,True,True,False
1,job_exposure.csv,3,11-1021,General and Operations Managers,0.1378,11-1021,True,True,False
2,job_exposure.csv,4,11-1031,Legislators,0.0000,11-1031,True,True,False
3,job_exposure.csv,5,Nov-11,Advertising and Promotions Managers,0.1731,Nov-11,False,False,True
4,job_exposure.csv,6,Nov-21,Marketing Managers,0.3195,Nov-21,False,False,True
5,job_exposure.csv,7,Nov-22,Sales Managers,0.0433,Nov-22,False,False,True
6,job_exposure.csv,8,Nov-32,Public Relations Managers,0.2315,Nov-32,False,False,True
7,job_exposure.csv,9,Nov-12,Administrative Services Managers,0.0000,Nov-12,False,False,True


## Flagged malformed-code examples

In [9]:
exposure_clean.loc[~exposure_clean["code_join_eligible"], [
    "_source_row", "occ_code_raw", "title", "observed_exposure",
    "code_format_valid", "code_join_eligible"
]].head(10)

,_source_row,occ_code_raw,title,observed_exposure,code_format_valid,code_join_eligible
3,5,Nov-11,Advertising and Promotions Managers,0.1731,False,False
4,6,Nov-21,Marketing Managers,0.3195,False,False
5,7,Nov-22,Sales Managers,0.0433,False,False
6,8,Nov-32,Public Relations Managers,0.2315,False,False
7,9,Nov-12,Administrative Services Managers,0.0000,False,False
8,10,Nov-13,Facilities Managers,0.1345,False,False
9,11,Nov-21,Computer and Information Systems Managers,0.1559,False,False
10,12,Nov-31,Financial Managers,0.3907,False,False
11,13,Nov-51,Industrial Production Managers,0.0132,False,False
12,14,Nov-61,Purchasing Managers,0.2004,False,False


## Row-count reconciliation

In [10]:
stage_changes.loc[stage_changes["stage"] == "job_exposure", [
    "stage", "input_rows", "output_rows", "rows_excluded",
    "new_flags", "failed_conversions", "summary"
]]

,stage,input_rows,output_rows,rows_excluded,new_flags,failed_conversions,summary
1,job_exposure,756,756,0,32,0,"Preserved all records; validated SOC codes and exposure fractions. Invalid codes are join-ineligible, not repaired."


**Summary:** All 756 records remain in the cleaned file. The 32 malformed occupation codes are visible and flagged rather than silently deleted or guessed. The 724 valid SOC codes are unique and can be joined safely.

# 3. BLS OEWS employment and wage data

The metropolitan and nonmetropolitan workbooks contain May 2025 employment and wage estimates. Symbols such as `**`, `*`, and `#` have different meanings and must not be converted to zero.

## Original data checks

In [11]:
oews_raw_msa = pd.read_excel(
    SOURCE / "MSA_M2025_dl.xlsx", sheet_name="MSA_M2025_dl", nrows=5
)
oews_raw_msa

,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,O_GROUP,TOT_EMP,EMP_PRSE,JOBS_1000,LOC_QUOTIENT,PCT_TOTAL,PCT_RPT,H_MEAN,A_MEAN,MEAN_PRSE,H_PCT10,H_PCT25,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,total,75070,0.0,1000.000,1.00,NaN,NaN,26.13,54360,0.6,11.97,14.90,21.32,29.92,45.19,24900,30980,44340,62230,93990,NaN,NaN
1,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,major,5470,2.1,72.874,1.02,NaN,NaN,53.94,112190,0.9,22.98,31.25,45.90,64.34,92.98,47790,64990,95480,133840,193390,NaN,NaN
2,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,detailed,40,10.0,0.542,0.41,NaN,NaN,99.14,206200,6.3,42.50,58.65,79.70,113.51,187.68,88400,121980,165770,236100,390370,NaN,NaN
3,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,detailed,2120,1.6,28.241,1.25,NaN,NaN,54.63,113630,1.3,21.04,27.76,41.44,67.13,101.14,43770,57740,86200,139630,210360,NaN,NaN
4,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-2021,Marketing Managers,detailed,140,6.5,1.848,0.73,NaN,NaN,61.98,128910,3.8,34.14,41.23,52.45,74.40,95.07,71010,85770,109100,154750,197740,NaN,NaN


In [12]:
oews_raw_nonmetro = pd.read_excel(
    SOURCE / "BOS_M2025_dl.xlsx", sheet_name="BOS_M2025_dl", nrows=5
)
oews_raw_nonmetro

,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,O_GROUP,TOT_EMP,EMP_PRSE,JOBS_1000,LOC_QUOTIENT,PCT_TOTAL,PCT_RPT,H_MEAN,A_MEAN,MEAN_PRSE,H_PCT10,H_PCT25,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,total,67570,0.0,1000.000,1.00,NaN,NaN,23.29,48440,0.4,11.33,14.78,19.71,26.99,36.87,23570,30750,41000,56130,76700,NaN,NaN
1,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,major,2700,1.6,39.982,0.56,NaN,NaN,51.72,107580,0.9,24.48,31.25,46.11,61.34,84.82,50920,65010,95910,127590,176420,NaN,NaN
2,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,detailed,40,14.5,0.539,0.41,NaN,NaN,79.16,164650,4.1,59.34,61.95,72.82,87.54,110.05,123440,128860,151460,182090,228910,NaN,NaN
3,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,detailed,750,4.5,11.137,0.49,NaN,NaN,59.06,122830,1.7,25.00,31.14,50.00,75.01,102.70,51990,64780,103990,156020,213610,NaN,NaN
4,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-2022,Sales Managers,detailed,110,17.6,1.698,0.41,NaN,NaN,64.22,133570,2.5,31.66,42.06,62.39,83.96,104.94,65860,87490,129770,174630,218270,NaN,NaN


In [13]:
marker_rows = []
for filename, columns in validation["oews"]["marker_counts"].items():
    for column, counts in columns.items():
        for marker, count in counts.items():
            marker_rows.append({
                "source_file": filename,
                "column": column,
                "source_marker": "blank" if marker == "" else marker,
                "count": count,
            })
marker_table = pd.DataFrame(marker_rows)
marker_table.head(12)

,source_file,column,source_marker,count
0,BOS_M2025_dl.xlsx,TOT_EMP,**,760
1,BOS_M2025_dl.xlsx,EMP_PRSE,**,760
2,BOS_M2025_dl.xlsx,JOBS_1000,**,760
3,BOS_M2025_dl.xlsx,LOC_QUOTIENT,**,760
4,BOS_M2025_dl.xlsx,PCT_TOTAL,blank,48689
5,BOS_M2025_dl.xlsx,PCT_RPT,blank,48689
6,BOS_M2025_dl.xlsx,H_MEAN,*,2952
7,BOS_M2025_dl.xlsx,H_MEAN,#,39
8,BOS_M2025_dl.xlsx,A_MEAN,*,435
9,BOS_M2025_dl.xlsx,A_MEAN,#,39


## Cleaning rules

- Add source file, sheet, source row, release, and area class.
- Append metropolitan and nonmetropolitan records with `pd.concat()`.
- Convert published numeric fields with `pd.to_numeric()`.
- Store each source marker's meaning in a matching `__status` column.
- Keep missing, unavailable, thresholded, and censored estimates as missing numeric values, not zero.
- Check wage percentile order after conversion.

## Cleaned data preview

In [14]:
oews_clean = pd.read_csv(OUTPUT / "cleaned_oews.csv.gz", nrows=8)
oews_clean[[
    "_source_file", "_source_row", "area_class", "AREA", "AREA_TITLE",
    "OCC_CODE", "OCC_TITLE", "TOT_EMP", "TOT_EMP__status",
    "A_MEAN", "A_MEAN__status"
]]

,_source_file,_source_row,area_class,AREA,AREA_TITLE,OCC_CODE,OCC_TITLE,TOT_EMP,TOT_EMP__status,A_MEAN,A_MEAN__status
0,BOS_M2025_dl.xlsx,2,nonmetropolitan,100001,Northwest Alabama nonmetropolitan area,00-0000,All Occupations,67570.0,reported,48440.0,reported
1,BOS_M2025_dl.xlsx,3,nonmetropolitan,100001,Northwest Alabama nonmetropolitan area,11-0000,Management Occupations,2700.0,reported,107580.0,reported
2,BOS_M2025_dl.xlsx,4,nonmetropolitan,100001,Northwest Alabama nonmetropolitan area,11-1011,Chief Executives,40.0,reported,164650.0,reported
3,BOS_M2025_dl.xlsx,5,nonmetropolitan,100001,Northwest Alabama nonmetropolitan area,11-1021,General and Operations Managers,750.0,reported,122830.0,reported
4,BOS_M2025_dl.xlsx,6,nonmetropolitan,100001,Northwest Alabama nonmetropolitan area,11-2022,Sales Managers,110.0,reported,133570.0,reported
5,BOS_M2025_dl.xlsx,7,nonmetropolitan,100001,Northwest Alabama nonmetropolitan area,11-3021,Computer and Information Systems Managers,60.0,reported,129620.0,reported
6,BOS_M2025_dl.xlsx,8,nonmetropolitan,100001,Northwest Alabama nonmetropolitan area,11-3031,Financial Managers,150.0,reported,145710.0,reported
7,BOS_M2025_dl.xlsx,9,nonmetropolitan,100001,Northwest Alabama nonmetropolitan area,11-3051,Industrial Production Managers,220.0,reported,113780.0,reported


## Cleaned examples of special source markers

In [15]:
oews_special = pd.read_csv(
    OUTPUT / "cleaned_oews.csv.gz",
    usecols=["_source_file", "_source_row", "OCC_CODE", "OCC_TITLE",
             "TOT_EMP", "TOT_EMP__status", "H_MEAN", "H_MEAN__status"]
)
oews_special.loc[
    oews_special["TOT_EMP__status"].notna() |
    oews_special["H_MEAN__status"].notna()
].head(10)

,_source_file,_source_row,OCC_CODE,OCC_TITLE,TOT_EMP,H_MEAN,TOT_EMP__status,H_MEAN__status
0,BOS_M2025_dl.xlsx,2,00-0000,All Occupations,67570.0,23.29,reported,reported
1,BOS_M2025_dl.xlsx,3,11-0000,Management Occupations,2700.0,51.72,reported,reported
2,BOS_M2025_dl.xlsx,4,11-1011,Chief Executives,40.0,79.16,reported,reported
3,BOS_M2025_dl.xlsx,5,11-1021,General and Operations Managers,750.0,59.06,reported,reported
4,BOS_M2025_dl.xlsx,6,11-2022,Sales Managers,110.0,64.22,reported,reported
5,BOS_M2025_dl.xlsx,7,11-3021,Computer and Information Systems Managers,60.0,62.32,reported,reported
6,BOS_M2025_dl.xlsx,8,11-3031,Financial Managers,150.0,70.05,reported,reported
7,BOS_M2025_dl.xlsx,9,11-3051,Industrial Production Managers,220.0,54.70,reported,reported
8,BOS_M2025_dl.xlsx,10,11-3061,Purchasing Managers,60.0,48.97,reported,reported
9,BOS_M2025_dl.xlsx,11,11-3121,Human Resources Managers,40.0,50.69,reported,reported


## Row-count reconciliation

In [16]:
stage_changes.loc[stage_changes["stage"] == "oews", [
    "stage", "input_rows", "output_rows", "rows_excluded",
    "failed_conversions", "summary"
]]

,stage,input_rows,output_rows,rows_excluded,failed_conversions,summary
2,oews,198712,198712,0,0,Appended MSA and BOS with area class; converted numeric fields and preserved marker meanings in status columns. No rows removed.


**Summary:** The two OEWS tables produce 198,712 cleaned records with no row loss and no failed numeric conversions. Marker meanings remain available in status columns, so an unavailable estimate is distinguishable from a real zero.

# 4. BLS employment projections

The workbook contains multiple tables with different grains. Each table is cleaned separately so that titles, footer rows, and numeric fields are handled without mixing table meanings.

## Original data checks

In [17]:
projection_book = pd.ExcelFile(SOURCE / "occupation.XLSX")
print("Workbook sheets:", projection_book.sheet_names)
projection_raw = pd.read_excel(
    SOURCE / "occupation.XLSX", sheet_name="Table 1.2", header=1, nrows=5
)
projection_raw

Workbook sheets: ['Index', 'Table 1.1', 'Table 1.2', 'Table 1.3', 'Table 1.4', 'Table 1.5', 'Table 1.6', 'Table 1.8', 'Table 1.9', 'Table 1.10', 'Table 1.11', 'Table 1.12']


,2025 National Employment Matrix title,2025 National Employment Matrix code,Occupation type,"Employment, 2025","Employment, 2035","Employment distribution, percent, 2025","Employment distribution, percent, 2035","Employment change, numeric, 2025–35","Employment change, percent, 2025–35","Percent self-employed, 2025","Occupational openings, 2025–35 annual average","Median annual wage, dollars, 2025",Typical education needed for entry,Work experience in a related occupation,Typical on-the-job training needed to attain competency in the occupation,Related Occupational Outlook Handbook (OOH) content,Related Occupational Outlook Handbook (OOH) content.1
0,"Total, all occupations",00-0000,Summary,170280.8,176198.5,100.0,100.0,5917.6,3.5,5.8,17461.8,50980,—,—,—,—,—
1,Management occupations,11-0000,Summary,13683.4,14527.6,8.0,8.2,844.2,6.2,14.4,1081.7,126520,—,—,—,—,—
2,Top executives,11-1000,Summary,3904.1,4095.2,2.3,2.3,191.1,4.9,2.7,305.1,108510,—,—,—,—,—
3,Chief executives,11-1011,Line item,291.6,300.9,0.2,0.2,9.3,3.2,26.6,19.1,213990,Bachelor's degree,5 years or more,NaN,OOH Content,"<a href=""https://www.bls.gov/ooh/management/top-executives.htm"">OOH Content</a>"
4,General and operations managers,11-1021,Line item,3599.0,3780.3,2.1,2.1,181.3,5.0,0.8,285.0,105770,Bachelor's degree,5 years or more,NaN,OOH Content,"<a href=""https://www.bls.gov/ooh/management/top-executives.htm"">OOH Content</a>"


## Cleaning rules

- Preserve workbook, sheet, and source-row lineage.
- Remove only worksheet title and footer rows that are not observations.
- Trim display titles while preserving a raw-title column.
- Convert numeric measures without changing thousands, dollars, percentages, or rates.
- Keep negative projected employment change because decline is meaningful.
- Save every cleaned table separately.

## Cleaned data preview

In [18]:
projection_clean = pd.read_csv(OUTPUT / "projections" / "table_1_2.csv")
projection_clean.head(8)

,_source_file,_source_sheet,_source_row,2025 National Employment Matrix title,2025 National Employment Matrix code,Occupation type,"Employment, 2025","Employment, 2035","Employment distribution, percent, 2025","Employment distribution, percent, 2035","Employment change, numeric, 2025–35","Employment change, percent, 2025–35","Percent self-employed, 2025","Occupational openings, 2025–35 annual average","Median annual wage, dollars, 2025",Typical education needed for entry,Work experience in a related occupation,Typical on-the-job training needed to attain competency in the occupation,ooh_label,ooh_link_raw,2025 National Employment Matrix title__raw,"Employment, 2025__status","Employment, 2035__status","Employment distribution, percent, 2025__status","Employment distribution, percent, 2035__status","Employment change, numeric, 2025–35__status","Employment change, percent, 2025–35__status","Percent self-employed, 2025__status","Occupational openings, 2025–35 annual average__status","Median annual wage, dollars, 2025__status"
0,occupation.XLSX,Table 1.2,3,"Total, all occupations",00-0000,Summary,170280.8,176198.5,100.0,100.0,5917.6,3.5,5.8,17461.8,50980.0,—,—,—,—,—,"Total, all occupations",reported,reported,reported,reported,reported,reported,reported,reported,reported
1,occupation.XLSX,Table 1.2,4,Management occupations,11-0000,Summary,13683.4,14527.6,8.0,8.2,844.2,6.2,14.4,1081.7,126520.0,—,—,—,—,—,Management occupations,reported,reported,reported,reported,reported,reported,reported,reported,reported
2,occupation.XLSX,Table 1.2,5,Top executives,11-1000,Summary,3904.1,4095.2,2.3,2.3,191.1,4.9,2.7,305.1,108510.0,—,—,—,—,—,Top executives,reported,reported,reported,reported,reported,reported,reported,reported,reported
3,occupation.XLSX,Table 1.2,6,Chief executives,11-1011,Line item,291.6,300.9,0.2,0.2,9.3,3.2,26.6,19.1,213990.0,Bachelor's degree,5 years or more,NaN,OOH Content,"<a href=""https://www.bls.gov/ooh/management/top-executives.htm"">OOH Content</a>",Chief executives,reported,reported,reported,reported,reported,reported,reported,reported,reported
4,occupation.XLSX,Table 1.2,7,General and operations managers,11-1021,Line item,3599.0,3780.3,2.1,2.1,181.3,5.0,0.8,285.0,105770.0,Bachelor's degree,5 years or more,NaN,OOH Content,"<a href=""https://www.bls.gov/ooh/management/top-executives.htm"">OOH Content</a>",General and operations managers,reported,reported,reported,reported,reported,reported,reported,reported,reported
5,occupation.XLSX,Table 1.2,8,"Advertising, marketing, promotions, public relations, and sales managers",11-2000,Summary,1234.5,1298.6,0.7,0.7,64.1,5.2,3.2,93.4,155780.0,—,—,—,—,—,"Advertising, marketing, promotions, public relations, and sales managers",reported,reported,reported,reported,reported,reported,reported,reported,reported
6,occupation.XLSX,Table 1.2,9,Advertising and promotions managers,11-2011,Line item,33.2,31.9,0.0,0.0,-1.2,-3.7,35.1,2.3,133660.0,Bachelor's degree,Less than 5 years,NaN,OOH Content,"<a href=""https://www.bls.gov/ooh/management/advertising-promotions-and-marketing-managers.htm"">OOH Content</a>",Advertising and promotions managers,reported,reported,reported,reported,reported,reported,reported,reported,reported
7,occupation.XLSX,Table 1.2,10,Marketing and sales managers,11-2020,Summary,1071.7,1129.9,0.6,0.6,58.1,5.4,2.6,81.2,158040.0,—,—,—,—,—,Marketing and sales managers,reported,reported,reported,reported,reported,reported,reported,reported,reported


## Row-count reconciliation

In [19]:
pd.DataFrame({
    "measure": ["Input worksheet rows", "Clean data rows", "Non-data title/footer rows"],
    "count": [
        validation["projections"]["input_rows"],
        validation["projections"]["data_rows"],
        validation["projections"]["nondata_rows"],
    ],
})

,measure,count
0,Input worksheet rows,4876
1,Clean data rows,4844
2,Non-data title/footer rows,32


**Summary:** The workbook contains 4,876 inspected rows. Thirty-two title or footer rows are documented as non-data, leaving 4,844 cleaned observations across the projection tables.

# 5. Data integration and merge verification

The analysis begins with California detailed-occupation, cross-industry OEWS records. National AI exposure and national employment outlook are added by exact SOC occupation code. The joins provide context; they do not create local causal estimates.

In [20]:
merge_plan = pd.DataFrame([
    {"left table": "California OEWS", "right table": "AI exposure", "join key": "OCC_CODE = occ_code", "validation": "many-to-one"},
    {"left table": "California OEWS + exposure", "right table": "Projection Table 1.2", "join key": "OCC_CODE = projection code", "validation": "many-to-one"},
])
merge_plan

,left table,right table,join key,validation
0,California OEWS,AI exposure,OCC_CODE = occ_code,many-to-one
1,California OEWS + exposure,Projection Table 1.2,OCC_CODE = projection code,many-to-one


## Integrated data preview

In [21]:
analysis = pd.read_csv(OUTPUT / "analysis_california_occupation_region.csv.gz")
print("Shape:", analysis.shape)
analysis[[
    "AREA", "AREA_TITLE", "area_class", "OCC_CODE", "OCC_TITLE",
    "observed_exposure", "TOT_EMP", "A_MEAN",
    "projected_employment_change_pct_2025_35", "screening_status",
    "screening_score", "screening_rank_within_region"
]].head(8)

Shape: (11847, 74)


,AREA,AREA_TITLE,area_class,OCC_CODE,OCC_TITLE,observed_exposure,TOT_EMP,A_MEAN,projected_employment_change_pct_2025_35,screening_status,screening_score,screening_rank_within_region
0,600003,North Coast Region of California nonmetropolitan area,nonmetropolitan,11-1011,Chief Executives,0.0333,270.0,171850.0,3.2,complete four-component score,54.063604,107.0
1,600003,North Coast Region of California nonmetropolitan area,nonmetropolitan,11-1021,General and Operations Managers,0.1378,1910.0,117730.0,5.0,complete four-component score,67.349823,44.0
2,600003,North Coast Region of California nonmetropolitan area,nonmetropolitan,11-2021,Marketing Managers,NaN,90.0,150420.0,6.9,insufficient component data,NaN,NaN
3,600003,North Coast Region of California nonmetropolitan area,nonmetropolitan,11-2022,Sales Managers,NaN,490.0,112110.0,4.5,insufficient component data,NaN,NaN
4,600003,North Coast Region of California nonmetropolitan area,nonmetropolitan,11-3012,Administrative Services Managers,NaN,220.0,106020.0,5.2,insufficient component data,NaN,NaN
5,600003,North Coast Region of California nonmetropolitan area,nonmetropolitan,11-3013,Facilities Managers,NaN,120.0,102800.0,4.6,insufficient component data,NaN,NaN
6,600003,North Coast Region of California nonmetropolitan area,nonmetropolitan,11-3021,Computer and Information Systems Managers,NaN,140.0,154190.0,15.8,insufficient component data,NaN,NaN
7,600003,North Coast Region of California nonmetropolitan area,nonmetropolitan,11-3031,Financial Managers,NaN,380.0,148630.0,9.7,insufficient component data,NaN,NaN


## Join and checksum verification

In [22]:
metrics = validation["analysis"]
pd.DataFrame({
    "check": [
        "California base rows", "Exposure matches", "Exposure unmatched",
        "Projection matches", "Projection unmatched", "California regions",
        "Employment before joins", "Employment after joins"
    ],
    "result": [
        metrics["base_rows"], metrics["exposure_matches"], metrics["exposure_unmatched"],
        metrics["projection_matches"], metrics["projection_unmatched"], metrics["areas"],
        metrics["employment_checksum_before"], metrics["employment_checksum_after"]
    ]
})

,check,result
0,California base rows,11847.0
1,Exposure matches,10145.0
2,Exposure unmatched,1702.0
3,Projection matches,11847.0
4,Projection unmatched,0.0
5,California regions,28.0
6,Employment before joins,17277510.0
7,Employment after joins,17277510.0


## Screening score

The score is calculated only when all four inputs are present. Within each region, it combines 40% observed AI exposure, 30% employment size, 15% lower annual wage as a transition-support proxy, and 15% weaker national outlook as a transition-support proxy. All components remain visible for review.

In [23]:
print("Eligible for scoring:", f'{metrics["screening_eligible_rows"]:,}')
print("Retained without a score:", f'{metrics["screening_ineligible_rows"]:,}')
print("Top-25 shortlist rows:", f'{metrics["shortlist_rows"]:,}')
analysis["screening_score"].describe().to_frame("screening_score")

Eligible for scoring: 9,829
Retained without a score: 2,018
Top-25 shortlist rows: 701


,screening_score
count,9829.000000
mean,50.142436
std,15.774662
min,12.119342
25%,38.740000
50%,49.546667
75%,60.830645
max,94.595324


**Summary:** The many-to-one joins preserve all 11,847 California base rows and preserve the employment checksum of 17,277,510 jobs. Missing exposure does not remove a record; it gives the record an incomplete screening status.

# 6. Final validation

These tables summarize row changes, duplicate checks, and missing-value changes. They make each major transformation auditable instead of relying only on the statement that the code ran.

## Cleaning decision log

Each row documents the applied rule, its reason, and the numerical check used to verify the result.

In [24]:
decision_log = stage_changes[[
    "stage", "summary", "input_rows", "output_rows", "rows_excluded",
    "new_flags", "failed_conversions"
]].copy()
decision_log = decision_log.rename(columns={"summary": "rule"})
decision_log["reason"] = "Preserve source meaning and make every change traceable."
decision_log["check"] = (
    "input=" + decision_log["input_rows"].astype(str)
    + "; output=" + decision_log["output_rows"].astype(str)
    + "; excluded=" + decision_log["rows_excluded"].astype(str)
    + "; flags=" + decision_log["new_flags"].astype(str)
    + "; failed conversions=" + decision_log["failed_conversions"].astype(str)
)
decision_log[["stage", "rule", "reason", "check"]]

,stage,rule,reason,check
0,claude,"Parsed dates, hierarchy level, and value; added metric units and metric-aware flags. No rows removed.",Preserve source meaning and make every change traceable.,input=1636573; output=1636573; excluded=0; flags=0; failed conversions=0
1,job_exposure,"Preserved all records; validated SOC codes and exposure fractions. Invalid codes are join-ineligible, not repaired.",Preserve source meaning and make every change traceable.,input=756; output=756; excluded=0; flags=32; failed conversions=0
2,oews,Appended MSA and BOS with area class; converted numeric fields and preserved marker meanings in status columns. No rows removed.,Preserve source meaning and make every change traceable.,input=198712; output=198712; excluded=0; flags=0; failed conversions=0
3,projections,"Separated worksheet title/footer rows, preserved sheet grains, trimmed display titles, and converted numeric measures without changing units.",Preserve source meaning and make every change traceable.,input=4876; output=4844; excluded=32; flags=0; failed conversions=0
4,analysis_scope_and_joins,"Selected California detailed cross-industry region records, made two validated many-to-one context joins, and calculated a transparent four-component within-region screening rank. Base records and employment checksum were preserved.",Preserve source meaning and make every change traceable.,input=198712; output=11847; excluded=186865; flags=2018; failed conversions=0


## Changes after each major transformation

In [25]:
stage_changes

,stage,input_rows,output_rows,rows_excluded,changed_cells,new_flags,failed_conversions,summary
0,claude,1636573,1636573,0,6546292,0,0,"Parsed dates, hierarchy level, and value; added metric units and metric-aware flags. No rows removed."
1,job_exposure,756,756,0,756,32,0,"Preserved all records; validated SOC codes and exposure fractions. Invalid codes are join-ineligible, not repaired."
2,oews,198712,198712,0,3775528,0,0,Appended MSA and BOS with area class; converted numeric fields and preserved marker meanings in status columns. No rows removed.
3,projections,4876,4844,32,4844,0,0,"Separated worksheet title/footer rows, preserved sheet grains, trimmed display titles, and converted numeric measures without changing units."
4,analysis_scope_and_joins,198712,11847,186865,80966,2018,0,"Selected California detailed cross-industry region records, made two validated many-to-one context joins, and calculated a transparent four-component within-region screening rank. Base records and employment checksum were preserved."


## Duplicate checks

In [26]:
duplicate_checks

,table,phase,exact_duplicate_excess,repeated_key_excess
0,claude_metrics,raw,0,0
1,job_exposure,raw,0,17
2,job_exposure_valid_codes,clean,0,0
3,oews_nonmetropolitan,clean,0,0
4,oews_metropolitan,clean,0,0
5,projection_table_1_1,clean,0,0
6,projection_table_1_2,clean,0,0
7,projection_table_1_3,clean,0,0
8,projection_table_1_4,clean,0,0
9,projection_table_1_5,clean,0,0


## Missing values before and after cleaning

In [27]:
missing_summary = missing_counts.pivot_table(
    index=["table", "column"], columns="phase", values="missing",
    aggfunc="first"
).reset_index()
missing_summary.head(20)

phase,table,column,clean,raw
0,claude_metrics,category_name,0.0,0.0
1,claude_metrics,date_end,0.0,0.0
2,claude_metrics,date_start,0.0,0.0
3,claude_metrics,geo_id,0.0,0.0
4,claude_metrics,geo_level,0.0,0.0
5,claude_metrics,hierarchy_level,0.0,0.0
6,claude_metrics,metric_id,0.0,0.0
7,claude_metrics,node_external_id,0.0,0.0
8,claude_metrics,node_name,0.0,0.0
9,claude_metrics,value,0.0,0.0


## Important categorical and numeric checks

In [28]:
pd.DataFrame({
    "measure": [
        "Unique California regions", "Unique occupations", "Eligible rows",
        "Ineligible rows", "Minimum score", "Maximum score"
    ],
    "result": [
        analysis["AREA"].nunique(), analysis["OCC_CODE"].nunique(),
        int(analysis["screening_eligible"].sum()),
        int((~analysis["screening_eligible"]).sum()),
        analysis["screening_score"].min(), analysis["screening_score"].max()
    ]
})

,measure,result
0,Unique California regions,28.000000
1,Unique occupations,783.000000
2,Eligible rows,9829.000000
3,Ineligible rows,2018.000000
4,Minimum score,12.119342
5,Maximum score,94.595324


## Five predeclared record checks

These five records were declared before the final run. They test the most important difficult cases: a malformed occupation code, a high but valid time metric, unavailable employment, a censored wage, and negative projected growth.

In [29]:
record_checks = pd.read_csv(OUTPUT / "record_checks.csv")
record_checks[[
    "id", "source_file", "source_sheet", "source_row", "field",
    "original", "actual", "actual_status", "passed", "reason"
]]

,id,source_file,source_sheet,source_row,field,original,actual,actual_status,passed,reason
0,R1,job_exposure.csv,NaN,5,occ_code,Nov-11,Nov-11,False,True,The date-like value is not a valid six-digit SOC code. Preserve it and do not guess a repair.
1,R2,aei_claude_ai_2026-06-26.csv,NaN,396056,value,199.11,199.11,False,True,A value above 100 is valid for a time metric; a global percentage rule would be wrong.
2,R3,BOS_M2025_dl.xlsx,BOS_M2025_dl,15,TOT_EMP,**,NaN,employment_estimate_unavailable,True,"Unavailable employment is unknown, not zero."
3,R4,BOS_M2025_dl.xlsx,BOS_M2025_dl,1711,H_MEAN,#,NaN,censored_exact_value_unavailable,True,A censored wage is not an exact value and must not be imputed.
4,R5,occupation.XLSX,Table 1.5,4,"Employment change, percent, 2025–35",-34.4,-34.4,NaN,True,Negative projected growth is meaningful and must remain negative.


In [30]:
assert record_checks["passed"].all()
print(
    f'{validation["record_checks_passed"]} of '
    f'{validation["record_checks_total"]} predeclared checks passed.'
)

5 of 5 predeclared checks passed.


## How one Codex suggestion was checked

Codex suggested preserving date-like occupation codes instead of guessing corrections. The original exposure CSV was read again without the cleaning helpers, an anchored six-digit SOC pattern was applied, and valid-code duplicates were counted independently. The result below supports accepting the suggestion.

In [31]:
pd.DataFrame([validation["codex_check"]])

,suggestion,independent_method,rows,invalid_codes,valid_codes,valid_code_duplicate_excess,conclusion,student_review_pending
0,Do not automatically repair date-like occupation codes; preserve and flag them before exact-code joins.,Re-read the original CSV without pipeline helpers; apply an anchored six-digit SOC pattern and count duplicate valid keys.,756,32,724,0,Accepted. The 32 malformed strings are not sufficient evidence for a unique repair; all 724 valid exact codes are unique.,False


# 7. Saved cleaned outputs

The full cleaned tables, audit files, and compact submission sample are saved without overwriting any original source file.

In [32]:
saved_outputs = ["final_cleaned_sample.csv"] + sorted(
    str(path.relative_to(ROOT))
    for path in OUTPUT.rglob("*")
    if path.is_file()
)
pd.DataFrame({"saved_file": saved_outputs})

,saved_file
0,final_cleaned_sample.csv
1,outputs\analysis_california_occupation_region.csv.gz
2,outputs\cleaned_claude_metrics.csv.gz
3,outputs\cleaned_job_exposure.csv
4,outputs\cleaned_oews.csv.gz
5,outputs\cleaned_sample_max_50.csv
6,outputs\duplicates.csv
7,outputs\exclusion_ledger.csv
8,outputs\missing_counts.csv
9,outputs\projections\table_1_1.csv


# 8. Difficult-case cleaned sample

This purposeful sample displays real rows selected to demonstrate high employment, high exposure, unmatched codes, special missing-value markers, negative outlook, and the high-but-valid Claude time value.

In [33]:
difficult_sample = pd.read_csv(OUTPUT / "cleaned_sample_max_50.csv")
difficult_sample[[
    "sample_type", "source_file", "source_row", "AREA_TITLE", "OCC_CODE",
    "OCC_TITLE", "TOT_EMP", "A_MEAN", "observed_exposure",
    "projected_employment_change_pct_2025_35", "original_value",
    "clean_value", "status", "selection_reason"
]]

,sample_type,source_file,source_row,AREA_TITLE,OCC_CODE,OCC_TITLE,TOT_EMP,A_MEAN,observed_exposure,projected_employment_change_pct_2025_35,original_value,clean_value,status,selection_reason
0,analysis_record,MSA_M2025_dl.xlsx,79222,"Los Angeles-Long Beach-Anaheim, CA",31-1120,Home Health and Personal Care Aides,385530.0,35890.0,NaN,18.1,NaN,NaN,NaN,high employment
1,analysis_record,MSA_M2025_dl.xlsx,79272,"Los Angeles-Long Beach-Anaheim, CA",35-3023,Fast Food and Counter Workers,160270.0,41080.0,0.0000,5.8,NaN,NaN,NaN,high employment
2,analysis_record,MSA_M2025_dl.xlsx,79328,"Los Angeles-Long Beach-Anaheim, CA",41-2031,Retail Salespersons,130650.0,43010.0,0.3222,-0.3,NaN,NaN,NaN,high employment
3,analysis_record,MSA_M2025_dl.xlsx,79630,"Los Angeles-Long Beach-Anaheim, CA",53-7062,"Laborers and Freight, Stock, and Material Movers, Hand",120720.0,44070.0,0.0000,1.8,NaN,NaN,NaN,high employment
4,analysis_record,MSA_M2025_dl.xlsx,121408,"San Francisco-Oakland-Fremont, CA",31-1120,Home Health and Personal Care Aides,119120.0,38810.0,NaN,18.1,NaN,NaN,NaN,high employment
5,analysis_record,MSA_M2025_dl.xlsx,79394,"Los Angeles-Long Beach-Anaheim, CA",43-9061,"Office Clerks, General",112270.0,50980.0,0.4504,-6.0,NaN,NaN,NaN,high employment
6,analysis_record,BOS_M2025_dl.xlsx,3283,North Coast Region of California nonmetropolitan area,15-1251,Computer Programmers,40.0,104610.0,0.7451,-7.3,NaN,NaN,NaN,high observed exposure
7,analysis_record,MSA_M2025_dl.xlsx,9721,"Bakersfield-Delano, CA",15-1251,Computer Programmers,100.0,NaN,0.7451,-7.3,NaN,NaN,NaN,high observed exposure
8,analysis_record,MSA_M2025_dl.xlsx,48486,"Fresno, CA",15-1251,Computer Programmers,110.0,98190.0,0.7451,-7.3,NaN,NaN,NaN,high observed exposure
9,analysis_record,MSA_M2025_dl.xlsx,78941,"Los Angeles-Long Beach-Anaheim, CA",15-1251,Computer Programmers,3400.0,108020.0,0.7451,-7.3,NaN,NaN,NaN,high observed exposure


# 9. Final cleaned data sample

The submission CSV contains the top-ranked complete-data occupation from each of the 28 California regions. The first ten rows below show the final column structure and values. All selected columns are displayed; Jupyter does not replace the middle columns with dots.

In [34]:
final_sample = pd.read_csv(ROOT / "final_cleaned_sample.csv")
print("Rows:", len(final_sample))
print("Regions:", final_sample["AREA"].nunique())
final_sample[[
    "AREA", "AREA_TITLE", "area_class", "OCC_CODE", "OCC_TITLE",
    "observed_exposure", "TOT_EMP", "A_MEAN",
    "projected_employment_change_pct_2025_35", "exposure_percentile",
    "employment_percentile", "lower_wage_need_percentile",
    "weaker_outlook_need_percentile", "screening_score",
    "screening_rank_within_region", "screening_status"
]].head(10)

Rows: 28
Regions: 28


,AREA,AREA_TITLE,area_class,OCC_CODE,OCC_TITLE,observed_exposure,TOT_EMP,A_MEAN,projected_employment_change_pct_2025_35,exposure_percentile,employment_percentile,lower_wage_need_percentile,weaker_outlook_need_percentile,screening_score,screening_rank_within_region,screening_status
0,12540,"Bakersfield-Delano, CA",metropolitan,43-9061,"Office Clerks, General",0.4504,5500.0,46540.0,-6.0,0.962025,0.979747,0.802532,0.921519,93.734177,1,complete four-component score
1,17020,"Chico, CA",metropolitan,43-9061,"Office Clerks, General",0.4504,1700.0,45690.0,-6.0,0.958848,0.983539,0.769547,0.944444,93.569959,1,complete four-component score
2,600006,Eastern Sierra-Mother Lode Region of California nonmetropolitan area,nonmetropolitan,43-9061,"Office Clerks, General",0.4504,1120.0,50130.0,-6.0,0.968889,0.966667,0.724444,0.953333,92.922222,1,complete four-component score
3,20940,"El Centro, CA",metropolitan,43-9061,"Office Clerks, General",0.4504,1100.0,46230.0,-6.0,0.958115,0.963351,0.738220,0.934555,92.316754,1,complete four-component score
4,23420,"Fresno, CA",metropolitan,43-9061,"Office Clerks, General",0.4504,8710.0,48530.0,-6.0,0.962389,0.986726,0.751106,0.915929,93.102876,1,complete four-component score
5,25260,"Hanford-Corcoran, CA",metropolitan,43-9061,"Office Clerks, General",0.4504,910.0,48110.0,-6.0,0.957317,0.969512,0.695122,0.942073,91.935976,1,complete four-component score
6,31080,"Los Angeles-Long Beach-Anaheim, CA",metropolitan,43-9061,"Office Clerks, General",0.4504,112270.0,50980.0,-6.0,0.971901,0.995041,0.771901,0.885124,93.582645,1,complete four-component score
7,32900,"Merced, CA",metropolitan,43-9061,"Office Clerks, General",0.4504,1250.0,48920.0,-6.0,0.957082,0.969957,0.695279,0.903433,91.362661,1,complete four-component score
8,33700,"Modesto, CA",metropolitan,43-9061,"Office Clerks, General",0.4504,3280.0,48600.0,-6.0,0.961988,0.976608,0.777778,0.915205,93.172515,1,complete four-component score
9,34900,"Napa, CA",metropolitan,41-2031,Retail Salespersons,0.3222,2150.0,45630.0,-0.3,0.896861,0.995516,0.887892,0.809417,91.199552,1,complete four-component score


# Remaining limitations

- The 32 malformed exposure occupation codes cannot be corrected without an authoritative source mapping, so they remain visible but join-ineligible.
- AI exposure and employment projections are national occupation context, while OEWS employment and wages are California regional estimates.
- Claude-use measures may describe global, country, or subregion activity and should not be interpreted as California causal effects.
- Missing or censored OEWS estimates remain unknown; rows missing any score component do not receive a screening score.
- Screening weights and the lower-wage and weaker-outlook proxies are planning judgments that require human review.

# Conclusion

The cleaned analysis preserves the published phenomena: source identifiers and units remain intact, special missing-value markers are not changed to zero, malformed codes are flagged rather than guessed, and the joins do not multiply rows or employment. The final score is a transparent screening aid for human workforce-planning review, not an automatic funding or automation decision.